[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_2_3/03_tag_2_3_lokale_minima_lernrate.ipynb)

# Tag 2/3 - 03 Lokale Minima, Lernrate und Decay

Nicht jede Loss-Funktion ist eine saubere Schüssel. Bei nichtkonvexen Funktionen gibt es mehrere Täler. Dieses Notebook nutzt eine künstliche 1D-Loss-Landschaft, die überall im Plotbereich positiv ist. Eine konstante Verschiebung nach oben ändert den Gradienten nicht, macht die Darstellung aber didaktisch sauberer.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
print("Setup abgeschlossen.")


In [ ]:
def raw_loss(w):
    return 0.05 * w**2 - 0.2 * w + 1.5 * np.sin(3.0 * w)


# Konstante Verschiebung: Der Loss wird positiv, der Gradient bleibt identisch.
offset_grid = np.linspace(-8, 8, 5000)
LOSS_OFFSET = -raw_loss(offset_grid).min() + 0.05


def loss(w):
    return raw_loss(w) + LOSS_OFFSET


def gradient(w):
    return 0.1 * w - 0.2 + 4.5 * np.cos(3.0 * w)


def effective_learning_rate(initial_lr, decay, step):
    # Parameter im Fokus: decay reduziert die Schrittgröße über die Zeit.
    return initial_lr / (1.0 + decay * step)


def descent_path(start=-4.0, learning_rate=0.02, decay=0.0, steps=100):
    w = start
    path = []
    for step in range(steps + 1):
        current_lr = effective_learning_rate(learning_rate, decay, step)
        path.append((step, w, loss(w), current_lr))
        w = w - current_lr * gradient(w)
        if abs(w) > 8:
            break
    return path


w_grid = np.linspace(-5.5, 4.0, 1000)
loss_grid = loss(w_grid)

local_minima = []
for i in range(1, len(w_grid) - 1):
    if loss_grid[i] < loss_grid[i - 1] and loss_grid[i] < loss_grid[i + 1]:
        local_minima.append((w_grid[i], loss_grid[i]))

print(f"Minimum im Plotbereich: {loss_grid.min():.3f}")
for w_min, l_min in local_minima:
    print(f"lokales Minimum: w={w_min: .2f}, loss={l_min: .3f}")


## Interaktiv: Lernrate und Decay

Eine hohe Lernrate kann aus einem flachen lokalen Minimum herausführen. Decay macht die Schritte später kleiner. Das hilft oft: am Anfang schneller suchen, am Ende stabiler landen.


In [ ]:
def local_minima_demo(learning_rate=0.02, decay=0.0, start=-4.0, steps=100):
    path = descent_path(start=start, learning_rate=learning_rate, decay=decay, steps=steps)
    path_w = np.array([row[1] for row in path])
    path_loss = np.array([row[2] for row in path])
    path_lr = np.array([row[3] for row in path])

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    axes[0].plot(w_grid, loss_grid, color="black", label="Loss-Landschaft")
    axes[0].plot(path_w, path_loss, "o-", color="tab:orange", markersize=4, label="Gradient-Descent-Pfad")
    axes[0].scatter([start], [loss(start)], color="blue", s=90, label="Start")
    for w_min, l_min in local_minima:
        axes[0].scatter([w_min], [l_min], color="green", s=55)
    axes[0].set_xlabel("Parameter w")
    axes[0].set_ylabel("Loss")
    axes[0].set_title(f"Ende w={path_w[-1]:.3f}, Loss={path_loss[-1]:.3f}")
    axes[0].legend()

    axes[1].plot(range(len(path_lr)), path_lr)
    axes[1].set_title("Effektive Lernrate")
    axes[1].set_xlabel("Schritt")
    axes[1].set_ylabel("Learning Rate")
    plt.tight_layout()
    plt.show()


widgets.interact(
    local_minima_demo,
    learning_rate=widgets.FloatSlider(value=0.02, min=0.001, max=0.5, step=0.001, readout_format=".3f", description="Lernrate"),
    decay=widgets.FloatSlider(value=0.0, min=0.0, max=0.20, step=0.005, readout_format=".3f", description="Decay"),
    start=widgets.FloatSlider(value=-4.0, min=-5.0, max=2.0, step=0.1, description="Start"),
    steps=widgets.IntSlider(value=100, min=5, max=180, step=5, description="Schritte"),
);


## Direkter Vergleich

Ohne Decay kann eine große Lernrate lange hin und her springen. Mit Decay wird aus einer groben Suche schrittweise eine feinere Suche.


In [ ]:
plt.figure(figsize=(9, 4.8))
plt.plot(w_grid, loss_grid, color="black", label="Loss-Landschaft")

for lr, decay, color in [(0.02, 0.0, "tab:orange"), (0.35, 0.0, "tab:purple"), (0.35, 0.04, "tab:green")]:
    path = descent_path(start=-4.0, learning_rate=lr, decay=decay, steps=100)
    path_w = np.array([row[1] for row in path])
    path_loss = np.array([row[2] for row in path])
    plt.plot(path_w, path_loss, "o-", markersize=3, color=color, label=f"lr={lr}, decay={decay}, Ende={path_loss[-1]:.2f}")

plt.xlabel("Parameter w")
plt.ylabel("Loss")
plt.title("Lernrate und Decay im Vergleich")
plt.legend()
plt.show()
